# Combining Reactive Behaviors with Reinforcement Learning

In Part 1 of the class, we saw how agents can learn behavior in order to maximize rewards they receive from the environment. In Part 2, we saw how to implement reactive behavior for agents, enabling them to e.g. forage for resouces or avoid danger. In this notebook, we will see how to integrate both concepts, where an agent learn how to select reactive behaviors in order to maximize rewards.

The current scene contains a single agent (blue square), 10 green objects (with subtype `"A"`) and 10 yellow objects (with subtype `"B"`). When reaching an object, the agent will consume it. Each object subtype will be associated with a reward, either positive or negative. Whenever it consumes an object of a given subtype, the agent will receive the reward associated with this subtype. Using a simplified version of Reinforcement Learning (RL), the agent will learn which reactive behavior to execute towards each type of object. We expect it to learn to be attracted by objects providing a positive reward and be repulsed by objects providing a negative reward. 

For this aim, we will define two versions of each of the four canonical Braitenberg behaviors (`fear`, `aggression`, `love` and `shyness`): one version targeting objects of subtype `"A"` and one version targeting objects of subtype `"B"` (hence 8 behaviors in total). The objective of the RL algorithm will be to learn which behaviors to activate according to the rewards it receives.

As usual, let's first connect this notebook to the simulator:

In [ ]:
from vivarium.controllers import VivariumController
controller = VivariumController.start_session(scene_name="reactive_rl", start_controller_thread=False)

Let's create an alias variable for the unique agent in the scene:

In [ ]:
agent = controller.agents[0]

## Reactive behaviors as RL actions

In standard RL, the agent learns a *policy* mapping its current *observation* of the environment to an *action* to execute, with the objective of maximizing cumulative reward over an episode. 

In what follows, we consider an approach that was proposed in previous paper (e.g. [here](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0234434) and [here](https://link.springer.com/chapter/10.1007/978-3-319-95972-6_19)), where the agent instead learn to select predefined reactive behaviors in order to maximize cumulative reward. In this framework, and action (in the RL sense) consists in the selection of two behaviors (one targeting each object subtype) to be executed for a number of time steps. The agent will learn to predict the cumulative reward received when executing these behaviors. In future episodes, it will select behaviors with higher predicted reward more often.

As mentioned above, we consider that the agent have access to 8 predefined reactive behaviors (each of the 4 canonical Braitenberg behaviors, targeting each of the 2 object subtypes). For this we define a function `get_behavior` that takes two arguments: 

- `behavior_name`: the name of a canonical Braitenberg behavior (e.g. `"fear"`)
- `sensed_subtype`: the subtype that the behavior senses (e.g. `"A"`).

You don't need to understand how this function is implemented below, only that calling e.g. `get_behavior("shyness", "B")` will return the function defining the `shyness` behavior toward all objects with subtype `"B"`. Then we will be able to attach this returned behavior to the agent, as we did in previous sessions.

In [ ]:
def get_behavior(behavior_name, sensed_subtype):
    """Returns a behavior function based on the specified behavior name and sensed subtype."""
    
    def fear(agent):
        left, right = agent.proximeters(sensed_entities=[sensed_subtype])
        left_wheel = left
        right_wheel = right
        return left_wheel, right_wheel
    
    def aggression(agent):
        left, right = agent.proximeters(sensed_entities=[sensed_subtype])
        left_wheel = right
        right_wheel = left
        return left_wheel, right_wheel
    
    def love(agent):
        left, right = agent.proximeters(sensed_entities=[sensed_subtype])
        left_wheel = 1 - left
        right_wheel = 1 - right   
        return left_wheel, right_wheel
    
    def shyness(agent):
        left, right = agent.proximeters(sensed_entities=[sensed_subtype])
        left_wheel = 1 - right
        right_wheel = 1 - left   
        return left_wheel, right_wheel

    name_to_behavior = dict(fear=fear, aggression=aggression, love=love, shyness=shyness)

    return name_to_behavior[behavior_name]

Then we define a list of available "actions" for our agent. We consider that an action for the RL algorithm corresponds to the choice of two among the four canonical Braitenberg behaviors: one to execute towards objects with subtype `"A"` and one to execute towards objects with subtype `"B"`. For instance an action could be: execute the `shyness` behavior towards objects of subtype `"A"` and the `aggression` behavior towards objects of subtype `"B"`. Intuitively, we can imagine executing this action (e.g. these two behaviors for some time) will do a good job at maximizing rewards if objects `"A"` provide negative rewards and objects `"B"` provide positive reward, and a bad job if it is the reverse. 

We define the list of actions with (you don't need to understand this code):

In [ ]:
actions = []

for behavior_A in ['fear', 'aggression', 'love', 'shyness']:
    for behavior_B in ['fear', 'aggression', 'love', 'shyness']:
        actions.append({'A': behavior_A, 'B': behavior_B})

Let's print the generated list of actions:

In [ ]:
actions

As we see above, we have generated 16 possible actions in total, corresponding to each possible pairs of behaviors targeting each of the two object's subtypes. 

More precisely, each item in the printed list above corresponds to an available action. For instance, the first one `{'A': 'fear', 'B': 'fear'}` means "execute the `fear` behavior towards objects `"A"` and the `fear` behavior towards objects `"B"`". When this action is executed, the agent will therefore be afraid of all objects in the scene. 


Then we define a function `set_behaviors` that takes as arguments an `agent` and an `action` and attach the corresponding behaviors to the agent: 

In [ ]:
def set_behaviors(agent, action):
    agent.detach_all_behaviors(stop_motors=True)
    for sensed_subtype, behavior_name in action.items():
        agent.attach_behavior(get_behavior(behavior_name, sensed_subtype), name=f'{behavior_name}_{sensed_subtype}')

## Rewards

Then we define the reward associated with each object subtype. Here we defined a reward of 1 for objects of subtype `"A"` and of -1 for objects of subtype `"B"`:

In [ ]:
subtype_to_reward = {'A': 1.0, 'B': -1.0}

Then we define a routine (i.e. a function that will be executed at each time step on the agent) that computes the current reward and accumulates it in the agent internal state `agent.internal.reward`. For this, we need to know which object subtype has been consumed by the agent. However, the current version of Vivarium does not provide this information (we can only know if an entity was consumed, not what is the subtype of the consumed entity). As a workaround, we instead detect it using proximeter information, considering that if a proximeter is highly activated (> 0.96) for a given entity type, the agent will very likely consume the sensed entity. The routine also changes the color of the agent to either green or red, indicating if the last reward received by the agent was positive (agent turns green) or negative (agent turns red).

In [ ]:
# Proximeter threshold at which we consider
# the agent will be consuming the object
proximeter_threshold = 0.96

def reward_routine(agent):
    
    # Compute reward based on proximeter readings and subtype_to_reward mapping
    reward = 0
    for subtype, r in subtype_to_reward.items():
        if max(agent.proximeters(sensed_entities=[subtype])) > proximeter_threshold:
            reward += r
    agent.internal.reward += reward
    
    # Switch agent color based on reward
    if reward > 0:
        agent.color = 'green'
    if reward < 0:
        agent.color = 'red'

Then we initialize the reward internal state at 0 and attach the above routine to the agent:

In [ ]:
agent.detach_all_routines()
agent.internal.reward = 0
agent.attach_routine(reward_routine)

## Reactive RL algorithm

### Action values

We propose a simple RL algorithm where the agent simply learns the value of each the 16 available actions by trial and error. The value of an action will correspond to the current estimate of the cumulative reward it provides when the agent execute it for some time. The agent will probabilistically execute actions with higher estimated values. 

Initially, the value of all actions is 0:

In [ ]:
import numpy as np

values = np.zeros(len(actions))

During the learning loop, the agent will update this value table according to the rewards it receives when executing different actions. 

Action selection is probabilistic: the agent will sample actions with higher values more frequently. For this we can define a softmax probability distribution: 

In [ ]:

def softmax(x, temperature=1.0):
    """
    Compute softmax values for each sets of scores in x.
    The argument x will correspond to the current value vector.
    The temperature argument controls the exploration-exploitation trade-off.
    """
    return np.exp(x / temperature) / np.sum(np.exp(x / temperature), axis=0)



### Learning loop

Finally, the learning loop is structured in *epochs* and *action steps*. At each epoch, the agent will sample an action, i.e. the behaviors to be executed towards each object subtype, according to the softmax distribution from its current value vector. The agent will execute this action (i.e. two behaviors) for `n_actions_steps=1000` time steps in the environment, potentially consuming objects on its way and receiving the associated rewards. Then it will update its value vector according to the sum of rewards it received during these `n_action_steps` time steps. This process will repeat `n_epochs=50` times. 

Progressively, the value vector should learn decent estimates of the sum of rewards received when executing a given action for `n_actions_steps` steps. In consequence, it will progressively select the actions with the higher estimates more and more, ultimately learning how to collect objects with positive rewards and avoid objects with negative rewards.

In [ ]:
n_action_steps = 1000
n_epochs = 50

# Softmax temperature
temperature = 1

# Learning rate for value updates
lr = 0.1

for epoch in range(n_epochs):
    
    # Compute action probabilities using softmax and select an action based on these probabilities
    probs = softmax(values, temperature)
    action_idx = np.random.choice(range(len(actions)), p=probs)
    
    # Configure agent behaviors based on the selected action
    set_behaviors(agent, actions[action_idx])
    
    # Re-initialize the internal reward to 0
    agent.internal.reward = 0
    
    # Execute the selected action for a specified number of steps 
    # The reward routine will accumulate the reward during these steps based on what the agent consumes
    for t in range(n_action_steps):
        controller.step()
    
    # Update the value of the selected action using a simple reward prediction error update rule
    values[action_idx] += lr * (agent.internal.reward - values[action_idx])
    
    # Print the results of the current epoch before moving to the next one
    print(f'Epoch: {epoch}')
    max_action_idx = np.argmax(values)
    print(f'Action with highest value: {actions[max_action_idx]}. Prob= {probs[max_action_idx]}')
    print(f'Chosen action: {actions[action_idx]}. Prob={probs[action_idx]}')
    print(f'Reward received during the epoch: {agent.internal.reward}')
    print('======================================')

# When the learning loop is finished, we can detach all behaviors to stop the agent from moving
agent.detach_all_behaviors(stop_motors=True)

Once the learning process is completed, we can look at the values the agent as learned for each action:

In [ ]:
for action, value in zip(actions, values):
    print(action, value)

If the learning worked as expected, actions that attracts the agent toward the rewarding green object should have the highest values, i.e. actions with either `aggression` or `love` behavior toward objects of subtype `"A"` (the ones providing positive rewards) and either `fear` or `shyness` behavior for objects of subtype `"B"` (the ones providing negative rewards). 

## Readapting to a change of reward associations

Now let's swich the rewards associated with each object subtype: the positive rewards will be provided by objects with subtypes `"B"` (the yellow ones) and the negative rewards will be provided by objects with subtypes `"A"` (the green ones):

In [ ]:
subtype_to_reward = {'A': -1.0, 'B': 1.0}

Let's relaunch our learning loop to see if the agent is able to adapt to this drastic environmental change. Note that the previously learned behaviors are still in place: with the new reward associations we have just defined, the agent currently tend to avoid rewarding objects and to be attracted by the ones providing negative rewards.

The learning loop below is the exact same one as previously, but we do **not** reset the value vector. Let's see if the agent can learn to switch to the correct behaviors:

In [ ]:
for epoch in range(n_epochs):
    # Print current epoch
    
    # Compute action probabilities using softmax and select an action based on these probabilities
    probs = softmax(values, temperature)
    action_idx = np.random.choice(range(len(actions)), p=probs)
    
    # Configure agent behaviors based on the selected action
    set_behaviors(agent, actions[action_idx])
    
    # Re-initialize the internal reward to 0
    agent.internal.reward = 0
    
    # Execute the selected action for a specified number of steps 
    # The reward routine will accumulate the reward during these steps based on what the agent consumes
    for t in range(n_action_steps):
        controller.step()
    
    # Update the value of the selected action using a simple reward prediction error update rule
    values[action_idx] += lr * (agent.internal.reward - values[action_idx])
    
    # Print the results of the current epoch before moving to the next one
    print(f'Epoch: {epoch}')
    max_action_idx = np.argmax(values)
    print(f'Max action={actions[max_action_idx]}. Prob= {probs[max_action_idx]}')
    print(f'Chosen action: {actions[action_idx]}. Prob={probs[action_idx]}')
    print(f'Reward: {agent.internal.reward}')
    print('======================================')

# When the learning loop is finished, we can detach all behaviors to stop the agent from moving
agent.detach_all_behaviors(stop_motors=True)

Once the learning process is completed, we can look at the new values the agent as learned for each action:

In [ ]:
for action, value in zip(actions, values):
    print(action, value)

If the learning worked as expected, the agent should have adapted its preferred actions according to the change in the reward associations. Actions that attract the agent toward the rewarding yellow object should have highest values, i.e. actions with either `aggression` or `love` behavior toward objects of subtype `"B"` (the ones now providing positive rewards) and either `fear` or `shyness` for objects of subtype `"A"` (the ones now providing negative rewards). 